In [102]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
from scipy.spatial import KDTree

# parsing dei dati e estrazione delle feature
Dobbiamo estrarre per ogni poligono:
* `var_bb`: la varianza della distanza dei punti dal bounding box
* `var_center`: la varianza della distanza dei punti dal centroide del poligono

dove:
* **il centroide**: e' punto che ha per coordinate la media delle x e delle y
* **bounding box**: e' il rettangolo di area minore tra tutti che contiene il poligono

note:
* `np.vstack`: fa lo stack degli array in verticale. 
* **oss**: fare `vstack` corrisponde a concatenare $N$ array e poi fare il reshape ad $(1,N)$

In [ ]:

def parse_2rawdf(s):
    rawdf = []
    with open(s, 'r') as f:
        for line in f:
            lnsplit, label = line.strip().split(';')

            lnsplit = lnsplit.split(',')
            label = int(label)

            coords = np.array([float(x) for x in lnsplit])
            x = coords[0::2]
            y = coords[1::2]

            # calcola il centroide 
            # oppure: mean = np.mean(coords, axis=0)
            # e poi: np.var(mean)
            mean_x = x.mean()
            mean_y = y.mean()

            # calcola le distanze dal centroide
            dist_center = np.sqrt((x-mean_x)**2 + (y-mean_y)**2)

            # limiti bounding box
            x_min, x_max = x.min(), x.max()
            y_min, y_max = y.min(), y.max()

            # distanze dei punti
            dist_left = x - x_min
            dist_right = x_max - x
            dist_up = y_max - y
            dist_down = y - y_min

            # distanze bb
            # fai lo stack delle distanze, poi trova per ogni punto (axis=0) il minimo
            dist_bb = np.min(np.vstack([dist_left, dist_right, dist_up, dist_down]), axis=0)

            var_bb = dist_bb.var()
            var_center = dist_center.var()

            rawdf.append({'coords': coords, 'var_center': var_center, 'var_bb': var_bb, 'label': label})
    return rawdf


#  split

In [104]:
def train_test_split(X, y, train_size = 0.6):
    etichette = np.unique(y)
    n = X.shape[0]

    train_idxs = []
    for lab in etichette:
        # a = np.where(y==lab)[0]
        a, = np.where(y==lab)
        n0 = a.shape[0]
        train_idxs.append( np.random.choice(a, size=int(n0*train_size), replace=False))
    train_idxs = np.concatenate(train_idxs)
    test_idxs = np.setdiff1d(np.arange(n), train_idxs)
    
    return X[train_idxs], y[train_idxs], X[test_idxs], y[test_idxs]

# KNN
copia e incolla dalle lezioni

In [105]:

def mode(a):
    itms, cnts = np.unique(np.array(a), return_counts=True )

    return itms[np.argmax(cnts)], max(cnts)

class KNN(object):
    def __init__(self, k = 5):
        self.k = k
        self.X = None
        self.y = None
        self.tree = None
    
    def fit(self, X, y):
        self.X = X
        self.y = y
        self.tree = KDTree(self.X)
        
    def predict(self, x):        
        # Ottiene gli indici dei k punti più vicini
        _, k_indices = self.tree.query(x, k=self.k)
        
        # Prende le etichette corrispondenti 
        # e ritorna l'etichetta piu frequente nei k piu vicini
        return mode(self.y[k_indices])
    
    def test(self, xt, l):
        pred = np.array([self.predict(x)[0] for x in xt] == l)
        print("Accuracy: ", pred.sum()/pred.shape[0])
        

# Esecuzione!

In [127]:

path = os.path.join('dataset', '01-01-quadrati.csv')
df = pd.DataFrame(parse_2rawdf(path))
X = df[["var_bb", "var_center"]].to_numpy()
y = df["label"].to_numpy()

X_train, y_train, X_test, y_test = train_test_split(X,y)

knn = KNN()
knn.fit(X_train,y_train)
knn.test(X_test,y_test)

Accuracy:  0.9875


# ottimizzazione e varie...
* `punti = np.fromstring(forma, sep=',')` permette di leggere facilmente i dati in input da una stringa. E poi posso fare il reshape!
* `punti = punti.reshape((-1,2))`
* `baricentro = np.mean(punti, axis=0)`